In [25]:
#TODO make sure this renders in the github repo

In [26]:
# - [ ] #TODO add info about warmup steps etc for each model.

# Configurations For The Llama 3 Models

![chart](./showcase_images/model_sizes_chart.png)

- **Layers:** Noted as **num_layers**. How many Decoder layers the model has.
- **Model Dimensions:** Also noted as **d_model** or $d_{model}$. It represents the expected features for each token in the input and output sequences.
  - Example: when the Token Embeddings layer happen, each token is converted into a dense vector of size $d_{model}$.
- **FFN Dimensions:** Noted as **d_ff**. Is the hidden size of the Feed Forward SwiGLU sub-layer. We use this to expand the linear layers for SwiGLU, which gives the model more parameters to learn non-linear functions. 
- **Attention Heads:** This the the total number of Query heads, note K & V use the Key/Value Heads hyperparameter.
  - If d_model = $4{,}096$ and attn_heads = $32$, each individual head processes a vector of size $4{,}096/32 = 128$ 
- **Key/Value Heads:** Noted as **num_kv_heads**. The number of key and value heads in the attention mechanism. The reason the number of Key and Value heads is smaller than the number of Query heads, is because this allows faster generation, larger batch sizes, and reduces the memory footprint.
- **Peak Learning Rate:** At the end of the warmup, the learning rate hits the peak it can be, then it starts to decay. This keeps training stable, and prevents exploding gradients early on when the weights are random.

- **Vocabulary Size:** The model's dictionary. It represents the words/tokens model can understand. It is static.
    - **Context Window length (context_len/sequence_len):** The model's short-term memory. Represents how much words/tokens the model can remember at one time. So, as you chat with the model, it loses track of earlier context that no longer fits its context window. In the data, it is the number of tokens in a single document/row in a list of documents.
      - **Micro Batch Size:** The number of sequences stacked together in a single batch.
      - Example: `micro_batch_size` $=4$ and `context_len` $=4{,}096$, a single batch will contain $4*4{,}096 = 16{,}384 \, \text{tokens}$.

- **Positional Embeddings** The frequency that the Q and K tokens are rotated by, to give them positional information.

The [Llama-3.1 8B tokenizer](https://huggingface.co/meta-llama/Llama-3.1-8B/blob/main/tokenizer_config.json) consists of $128{,}255$ tokens.
- **Standard tokens:** IDs $0$ through $127{,}999$ ($128{,}000$ total) are the standard tokens.
- **Special tokens:** IDs $128{,}000$ through $128{,}255$ ($256$ total) are the special tokens.
- **From Llama 3 paper:** "We use a vocabulary with 128K tokens. Our token vocabulary combines 100K tokens from the tiktoken tokenizer with 28K additional tokens to better support non-English languages. Compared to the Llama 2 tokenizer, our new tokenizer improves compression rates on a sample of English data from 3.17 to 3.94 characters per token. This enables the model to “read” more text for the same amount of training compute. We also found that adding 28K tokens from select non-English languages improved both compression ratios and downstream performance, with no impact on English tokenization."

**Token Budget:**
- In LLM, data is not measured in gigabytes, sentence pairs, or epochs. Compute and training length are measured in tokens. The token budget dictates the maximum number of tokens you want the model to process before training terminates:

$$\text{Token Budget} = \text{Global Batch Size(in tokens)} * \text{Total Training Steps}$$

Example: $\text{Token Budget} = 1.5 \text{Billion}$ tokens, and your hardware can handle a $\text{Global Batch Size}$ of $500{,}000$ tokens per step, then: $1.5 \text{Billion} / 500{,}000 = 3{,}000$ total training steps.

- **Total Training Steps**: The number of times the model's weights are updated.
- **Global Batch Size**: The total number of tokens the optimizer needs to look at together to calculate an accurate gradient. In code, it dictates when `optimizer.step()` is called.
  - Without **Gradient Accumulation**: $\text{Global Batch Size} = \text{Micro Batch Size} \times \text{Context Length}$
  - With **Gradient Accumulation**: $\text{Global Batch Size} = \text{Micro Batch Size} \times \text{Context Length} \times \text{Accumulation Steps}$
    - **Gradient Accumulation Steps**: The number of sequential forward and backward passes over a micro-batch the model executes before aggregating the gradients and updating the weights via `optimizer.step()`. This allows you to simulate a massive Global Batch Size that would otherwise not fit in your hardware's VRAM.
  - For this implementation, I will use gradient accumulation. And you only ever have to worry about the `micro_batch_size`, the config will handle setting the `total_training_steps` and `global_batch_size` based on the `micro_batch_size`, `context_len`, and `gradient_accumulation_steps`, and `token_budget`.

- **Micro Batch Size:** The number of sequences your hardware can hold in its VRAM (GPU Memory) during a single forward and backward pass.

In [ ]:
import os
import EasyJupyter
from pathlib import Path
import torch


class BaseConfig:
    """
    The is the base config class that all models inherit from.
    It should never be instantiated, only its child classes should be!

    The parameters are defined in the markdown cells of Llama_config.ipynb
    """

    # All Llama models use the same below parameters, except for my scale down model.
    num_kv_heads = 8 #TODO what about my scaled down model?
    activation_function = "SwiGLU"
    pos_embeddings_freq = 500_000  # RoPE #TODO what about my scaled down model?

    # Force child classes to implement the below attributes
    _required_attributes: dict[str, type] = {
        "model_name": str,
        "context_len": int,
        "num_layers": int,  # How many decoder layers the model has.
        "d_model": int,
        "d_ff": int,  # How much to expand the SwiGLU linear to.
        "attn_heads": int,  # The number of independent "Query" viewpoints the attention mechanism splits the data into.
        "rms_norm_eps": float,  # RMSNorm epsilon #TODO set to 1e-5 for most models, maybe higher for my scale down model.
        "peak_lr": float,  # The peak learning rate.
        "micro_batch_size": int,
        "token_budget": int,
        "gradient_accumulation_steps": int,
        "vocab_size":int
    }

    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        for attr, expected_type in cls._required_attributes.items():
            value = getattr(cls, attr, None)
            if value == "# TODO" or value == "#TODO":
                continue
            elif value is None:
                raise ValueError(f"Missing attribute {attr} in {cls.__name__}")
            elif not isinstance(value, expected_type):
                raise TypeError(
                    f"Attribute {attr} in {cls.__name__} must be of type {expected_type}, but got {type(value)}"
                )
        pass




    # ================== Training ==================
    ### 🚨 NOTE Continue Training from a checkpoint:
    #       Models I train are saved to ./model/checkpoints/chpt_dir_name
    #       That directory will contain the model checkpoint (.pt), its tokenizer (.json), and other files associated with that specific model.
    #       Also, you must use the same config that was used to train the model to load it!
    # TODO when saving the tokenizer and the model, add a file that just decsribes it, and its config, for example, when user commnads to train, ask them in the CLI for a short summary, then save the model config, and its summanry to a file in that chpt_dir_name
    # TODO remove this need for config to be named in the config, example: `python train.py --resume_from checkpoints/step_1000`
    continue_from_chpt: bool = False  # Continue training the model from a check point
    chpt_dir_name = "scaled_down_llama"

    warmup_steps = 8000  # TODO: scale down for my scaled down model




    # ================== Dataset ==================
    # TODO remove any of the below commented out attributes if not needed.
    # max_batch_seq_tokens = 8_000 # Paper: 25_000. Applies sequence limit to an entire batch of sequences.
    # dataset_name = "" #TODO

    special_tokens = {
        "pad_token": {
            "token": "<|pad_token|>",
            "ID": 0,
        },
        "doc_begin_token": {  # The token for the beginning of the text.
            "token": "<|begin_of_doc|>",
            "ID": 1,  # TODO is this necessary? Check if I used it
        },
        "doc_end_token": {  # The token for the end of document, it is injected between documents.
            "token": "<|end_of_doc|>",
            "ID": 2,
        },
        "unk_token": {  # The unknown token.
            "token": "<|unk|>",
            "ID": 3,
        },
    }

    # ================== System ==================
    num_workers = 0  # 🚨 NOTE: For NVIDIA GPUs the Golden Rule is: num_worker = 4 * num_GPU | On Mac Silicone even though I have a 32 core GPU, it is still only one GPU, best to set num_workers = 0.
    if torch.cuda.is_available():
        device = torch.device("cuda")
    elif torch.backends.mps.is_available():
        device = torch.device("mps")
    else:
        device = torch.device("cpu")

    def __init__(self):
        # ================== Folder Structure ==================
        self.PROJECT_ROOT = self._find_root()
        print("\nProject Root:", self.PROJECT_ROOT)
        self.DATA_DIR = self.PROJECT_ROOT / "data"
        self.MODEL_DIR = self.PROJECT_ROOT / "model"
        self.folders_to_make = [
            self.DATA_DIR,
            self.MODEL_DIR / "saved_models",
            self.MODEL_DIR / "saved_models",
            self.MODEL_DIR / "saved_models" / "pre-trained",  # Store Meta-models here
            self.MODEL_DIR / "checkpoints",
        ]
        self.CHPT_DIR = self.MODEL_DIR / "checkpoints"

    def _find_root(self) -> Path:
        """Find the root directory of the project, by climbing up the directory tree"""
        curr = Path.cwd().resolve()

        for parent in [curr] + list(curr.parents):
            # if (parent / ".gitignore").exists():
            #     return parent
            if parent.name == "How_to_build_an_LLM":
                return parent

        return curr

    def print_config(self):
        # TODO Make the print method in the parent and have the child call it to prints its own config.
        pass

    def _setup_folders(self):
        for folder in self.folders_to_make:
            folder.mkdir(parents=True, exist_ok=True)

    @property
    def global_micro_batch_size_tokens(self) -> int:
        """
        Calculates the total tokens processed per optimizer step.
        Formula: micro_batch_size * context_len * gradient_accumulation_steps
        """
        return self.micro_batch_size * self.context_len * self.gradient_accumulation_steps

    @property
    def total_training_steps(self) -> int:
        """
        Calculates the total number of optimizer steps required to spend the token budget.
        """
        return self.token_budget // self.global_micro_batch_size_tokens

In [ ]:
class Llama3_8B(BaseConfig):
    """
    Llama 3.1 8B Configuration. This model is not multi-modal, it only takes in text!
    """

    model_name = "Llama 3.1 8B"
    num_layers = 32
    d_model = 4_096
    d_ff = 14_336
    attn_heads = 32
    peak_lr = 3e-4
    rms_norm_eps= 1e-5 #TODO set to paper value
    gradient_accumulation_steps = "#TODO"
    context_len = "#TODO" # find the right value for this from the paper
    token_budget = "#TODO" # find the right value for this from the paper
    micro_batch_size = "#TODO" # find the right value for this from the paper
    vocab_size = "#TODO"

    def __init__(self):
        super().__init__()

In [ ]:
class Llama3_scaled_down(BaseConfig):
    """
    Scaled down version of the Llama 3 architecture that is trainable on my Mac.
    """
    # TODO play around with the hyperparameters to see what my mac can handle.
    model_name = "Scaled down Llama 3"
    num_layers = 6
    d_model = 256
    d_ff = 1024
    attn_heads = 32
    peak_lr = 3e-4
    micro_batch_size = 8
    rms_norm_eps= 1e-5 #TODO set to 
    context_len = 4096
    gradient_accumulation_steps = 125
    token_budget = 16
    vocab_size = "#TODO"

    # TODO for this model the basemodel configs need to be adjusted like vocab_size it to be somewhere between 16,000 to 32,000

    def __init__(self):
        super().__init__()
        print(os.getcwd())

In [30]:
# @i-c
l = Llama3_8B()
l._setup_folders()


Project Root: /Users/tonyavis/Main/AI_projects_and_res/How_to_build_an_LLM


In [31]:
# TODO add the config for the larger models